# 📊 Visualización del Dataset de Clientes (Sin Outliers)

Este notebook carga el archivo `customer_behavior_dataset_sin_outliers.csv`, renombra las columnas usando las descripciones del archivo `descripcion_de_columnas.txt`, y genera una tabla interactiva con búsqueda, ordenamiento y paginación.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
from datetime import datetime

## 1. Cargar descripciones de columnas

In [ ]:
# Parsear las descripciones desde el archivo de texto
column_mapping = {}
with open("descripcion_de_columnas.txt", "r", encoding="utf-8") as f:
    for line in f:
        if ":" in line:
            parts = line.split(":", 1)
            col_name = parts[0].strip()
            col_desc = parts[1].strip().rstrip(".")
            column_mapping[col_name] = col_desc

# Agregar columna creada dinámicamente
column_mapping['Edad'] = 'Edad del cliente'

print(f"Se cargaron {len(column_mapping)} descripciones de columnas.")
for k, v in list(column_mapping.items())[:5]:
    print(f"  {k} → {v}")
print("  ...")

## 2. Cargar el dataset limpio y renombrar columnas

In [ ]:
# Cargar CSV
df = pd.read_csv("customer_behavior_dataset_sin_outliers.csv")

# Construir mapeo solo para columnas que existen en el dataset
rename_map = {col: column_mapping.get(col, col) for col in df.columns}
df_renombrado = df.rename(columns=rename_map)

print(f"Dataset cargado: {df_renombrado.shape[0]} filas × {df_renombrado.shape[1]} columnas")
print(f"\nColumnas renombradas:")
for orig, nuevo in rename_map.items():
    indicador = "✅" if orig != nuevo else "⚠️ (sin cambio)"
    print(f"  {indicador} {orig} → {nuevo}")

## 3. Resumen de métricas clave

In [ ]:
# Métricas rápidas
col_ingreso = rename_map.get('Income', 'Income')
col_edad = rename_map.get('Edad', 'Edad')

ingreso_prom = df_renombrado[col_ingreso].mean() if col_ingreso in df_renombrado.columns else None
edad_prom = df_renombrado[col_edad].mean() if col_edad in df_renombrado.columns else None

metrics_html = f"""
<div style="display: flex; gap: 20px; flex-wrap: wrap; margin: 20px 0;">
    <div style="flex: 1; min-width: 200px; background: linear-gradient(135deg, #1e293b, #0f172a); border: 1px solid #334155; border-radius: 16px; padding: 24px; text-align: center;">
        <div style="font-size: 14px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px;">Clientes Registrados</div>
        <div style="font-size: 36px; font-weight: 700; background: linear-gradient(135deg, #3b82f6, #8b5cf6); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">{df_renombrado.shape[0]:,}</div>
    </div>
    <div style="flex: 1; min-width: 200px; background: linear-gradient(135deg, #1e293b, #0f172a); border: 1px solid #334155; border-radius: 16px; padding: 24px; text-align: center;">
        <div style="font-size: 14px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px;">Variables / Columnas</div>
        <div style="font-size: 36px; font-weight: 700; background: linear-gradient(135deg, #3b82f6, #8b5cf6); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">{df_renombrado.shape[1]}</div>
    </div>
    <div style="flex: 1; min-width: 200px; background: linear-gradient(135deg, #1e293b, #0f172a); border: 1px solid #334155; border-radius: 16px; padding: 24px; text-align: center;">
        <div style="font-size: 14px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px;">Ingreso Promedio</div>
        <div style="font-size: 36px; font-weight: 700; background: linear-gradient(135deg, #3b82f6, #8b5cf6); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">${ingreso_prom:,.2f}</div>
    </div>
    <div style="flex: 1; min-width: 200px; background: linear-gradient(135deg, #1e293b, #0f172a); border: 1px solid #334155; border-radius: 16px; padding: 24px; text-align: center;">
        <div style="font-size: 14px; color: #94a3b8; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 8px;">Edad Promedio</div>
        <div style="font-size: 36px; font-weight: 700; background: linear-gradient(135deg, #3b82f6, #8b5cf6); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">{edad_prom:.1f} años</div>
    </div>
</div>
"""
display(HTML(metrics_html))

## 4. Tabla interactiva completa

La siguiente celda genera una tabla HTML interactiva con:
- 🔍 **Buscador** en tiempo real
- ⬆️⬇️ **Ordenamiento** por columna (clic en el encabezado)
- 📄 **Paginación** ajustable (10, 25, 50 o 100 filas)
- 🏷️ **Badges** para columnas binarias (campañas, quejas, respuesta)
- 💰 **Formato monetario** para columnas de ingresos y gastos

In [ ]:
import json

# Convertir datos a JSON para inyectar en el HTML
datos_json = df_renombrado.to_json(orient="records", force_ascii=False)
columnas_json = json.dumps(list(df_renombrado.columns), ensure_ascii=False)

tabla_html = f"""
<div id="tabla-container" style="font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;">
    <style>
        #tabla-container * {{ box-sizing: border-box; }}

        #tabla-container .controls {{
            display: flex; flex-wrap: wrap; gap: 16px; align-items: center;
            margin-bottom: 16px; padding: 16px;
            background: #1e293b; border: 1px solid #334155; border-radius: 12px;
        }}

        #tabla-container .search-box {{
            flex: 1; min-width: 250px; padding: 10px 14px;
            background: #0f172a; border: 1px solid #475569; border-radius: 8px;
            color: #e2e8f0; font-size: 14px; outline: none;
        }}
        #tabla-container .search-box:focus {{ border-color: #3b82f6; box-shadow: 0 0 0 3px rgba(59,130,246,0.2); }}
        #tabla-container .search-box::placeholder {{ color: #64748b; }}

        #tabla-container select {{
            padding: 10px 14px; background: #0f172a; border: 1px solid #475569;
            border-radius: 8px; color: #e2e8f0; font-size: 14px; cursor: pointer; outline: none;
        }}

        #tabla-container .table-wrap {{
            overflow-x: auto; border: 1px solid #334155; border-radius: 12px;
            background: #0f172a;
        }}
        #tabla-container .table-wrap::-webkit-scrollbar {{ height: 8px; }}
        #tabla-container .table-wrap::-webkit-scrollbar-track {{ background: #1e293b; }}
        #tabla-container .table-wrap::-webkit-scrollbar-thumb {{ background: #475569; border-radius: 4px; }}

        #tabla-container table {{
            width: 100%; border-collapse: collapse; font-size: 13px;
        }}

        #tabla-container th {{
            position: sticky; top: 0; z-index: 2;
            background: #1e293b; color: #e2e8f0; padding: 12px 16px;
            text-align: left; font-weight: 600; white-space: nowrap;
            border-bottom: 2px solid #334155; cursor: pointer; user-select: none;
        }}
        #tabla-container th:hover {{ background: #334155; }}

        #tabla-container td {{
            padding: 10px 16px; border-bottom: 1px solid rgba(51,65,85,0.5);
            color: #94a3b8; white-space: nowrap;
        }}
        #tabla-container tr:hover td {{ background: rgba(59,130,246,0.05); color: #e2e8f0; }}

        #tabla-container .badge {{
            display: inline-block; padding: 3px 10px; border-radius: 999px;
            font-size: 11px; font-weight: 600;
        }}
        #tabla-container .badge-yes {{ background: rgba(16,185,129,0.15); color: #10b981; border: 1px solid rgba(16,185,129,0.3); }}
        #tabla-container .badge-no {{ background: rgba(244,63,94,0.1); color: #f43f5e; border: 1px solid rgba(244,63,94,0.2); }}

        #tabla-container .pag-container {{
            display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap;
            padding: 14px 16px; background: #1e293b; border-top: 1px solid #334155;
            border-radius: 0 0 12px 12px; gap: 12px;
        }}
        #tabla-container .pag-info {{ font-size: 13px; color: #94a3b8; }}
        #tabla-container .pag-btns {{ display: flex; gap: 6px; }}
        #tabla-container .pag-btn {{
            padding: 6px 12px; background: rgba(255,255,255,0.05); border: 1px solid #334155;
            border-radius: 6px; color: #e2e8f0; font-size: 13px; cursor: pointer;
        }}
        #tabla-container .pag-btn:hover:not(:disabled) {{ background: linear-gradient(135deg, #3b82f6, #8b5cf6); border-color: transparent; }}
        #tabla-container .pag-btn:disabled {{ opacity: 0.3; cursor: not-allowed; }}
        #tabla-container .pag-btn.active {{ background: linear-gradient(135deg, #3b82f6, #8b5cf6); border-color: transparent; font-weight: 700; }}

        #tabla-container .no-results {{
            padding: 40px; text-align: center; color: #64748b; font-size: 15px;
        }}
    </style>

    <div class="controls">
        <input type="text" class="search-box" id="nb-search" placeholder="🔍 Buscar por ID, educación, estado civil...">
        <label style="color: #94a3b8; font-size: 13px;">Mostrar:</label>
        <select id="nb-page-size">
            <option value="10">10 filas</option>
            <option value="25" selected>25 filas</option>
            <option value="50">50 filas</option>
            <option value="100">100 filas</option>
        </select>
        <span style="font-size: 13px; color: #8b5cf6;">← → Deslizá horizontal para ver más columnas</span>
    </div>

    <div class="table-wrap" style="max-height: 500px; overflow-y: auto;">
        <table>
            <thead><tr id="nb-headers"></tr></thead>
            <tbody id="nb-body"></tbody>
        </table>
        <div id="nb-no-results" class="no-results" style="display:none;">No se encontraron registros.</div>
    </div>

    <div class="pag-container">
        <div class="pag-info" id="nb-pag-info"></div>
        <div class="pag-btns" id="nb-pag-btns"></div>
    </div>
</div>

<script>
(function() {{
    const rawData = {datos_json};
    const columns = {columnas_json};

    // Detectar columnas binarias (campañas, quejas, respuesta)
    const binaryCols = new Set();
    const moneyCols = new Set();
    columns.forEach(c => {{
        const cl = c.toLowerCase();
        if (cl.includes('campaña') || cl.includes('queja') || cl.includes('aceptó') || cl.includes('oferta') || cl === 'response') binaryCols.add(c);
        if (cl.includes('ingreso') || cl.includes('monto') || cl.includes('gasto')) moneyCols.add(c);
    }});

    let data = [...rawData];
    let filtered = [...rawData];
    let page = 1, size = 25, sortCol = null, sortDir = 'asc';

    function init() {{
        const hr = document.getElementById('nb-headers');
        hr.innerHTML = '';
        columns.forEach(c => {{
            const th = document.createElement('th');
            th.innerHTML = c + ' <span style="margin-left:6px;color:#64748b;font-size:11px;" id="si-' + c.replace(/\s/g,'_') + '">↕</span>';
            th.onclick = () => sort(c);
            hr.appendChild(th);
        }});
        document.getElementById('nb-search').oninput = e => {{ page = 1; filterSort(e.target.value.toLowerCase()); }};
        document.getElementById('nb-page-size').onchange = e => {{ size = +e.target.value; page = 1; render(); }};
        render();
    }}

    function sort(col) {{
        if (sortCol === col) sortDir = sortDir === 'asc' ? 'desc' : 'asc';
        else {{ sortCol = col; sortDir = 'asc'; }}
        columns.forEach(c => {{
            const el = document.getElementById('si-' + c.replace(/\s/g,'_'));
            if (el) {{ el.innerHTML = c === col ? (sortDir === 'asc' ? '▲' : '▼') : '↕'; el.style.color = c === col ? '#3b82f6' : '#64748b'; }}
        }});
        doSort();
        render();
    }}

    function filterSort(term) {{
        filtered = term ? rawData.filter(r => columns.some(c => r[c] != null && String(r[c]).toLowerCase().includes(term))) : [...rawData];
        doSort();
        render();
    }}

    function doSort() {{
        if (!sortCol) return;
        filtered.sort((a, b) => {{
            let va = a[sortCol], vb = b[sortCol];
            if (va == null) va = '';
            if (vb == null) vb = '';
            const na = Number(va), nb = Number(vb);
            if (!isNaN(na) && !isNaN(nb) && va !== '' && vb !== '') return sortDir === 'asc' ? na - nb : nb - na;
            const sa = String(va).toLowerCase(), sb = String(vb).toLowerCase();
            return sortDir === 'asc' ? sa.localeCompare(sb) : sb.localeCompare(sa);
        }});
    }}

    function render() {{
        const tbody = document.getElementById('nb-body');
        const noRes = document.getElementById('nb-no-results');
        tbody.innerHTML = '';
        const total = filtered.length;
        if (!total) {{ noRes.style.display = 'block'; document.getElementById('nb-pag-info').textContent = '0 registros'; document.getElementById('nb-pag-btns').innerHTML = ''; return; }}
        noRes.style.display = 'none';
        const pages = Math.ceil(total / size);
        if (page > pages) page = pages;
        const start = (page - 1) * size, end = Math.min(start + size, total);
        const slice = filtered.slice(start, end);

        slice.forEach(row => {{
            const tr = document.createElement('tr');
            columns.forEach(col => {{
                const td = document.createElement('td');
                let v = row[col];
                if (v == null) {{ td.innerHTML = '<span style="color:rgba(255,255,255,0.15)">-</span>'; }}
                else if (binaryCols.has(col)) {{
                    td.innerHTML = v == 1 ? '<span class="badge badge-yes">Sí</span>' : '<span class="badge badge-no">No</span>';
                }} else if (moneyCols.has(col)) {{
                    const n = Number(v);
                    if (!isNaN(n)) {{ td.textContent = '$' + n.toLocaleString('es-AR'); td.style.color = '#3b82f6'; td.style.fontWeight = '500'; }}
                    else td.textContent = v;
                }} else if (typeof v === 'number') {{ td.textContent = v.toLocaleString('es-AR'); }}
                else td.textContent = v;
                tr.appendChild(td);
            }});
            tbody.appendChild(tr);
        }});

        document.getElementById('nb-pag-info').textContent = `Mostrando ${{start+1}} a ${{end}} de ${{total}} registros`;

        // Paginación
        const bc = document.getElementById('nb-pag-btns');
        bc.innerHTML = '';
        const mkBtn = (label, pg, disabled, active) => {{
            const b = document.createElement('button');
            b.className = 'pag-btn' + (active ? ' active' : '');
            b.textContent = label;
            b.disabled = disabled;
            b.onclick = () => {{ page = pg; render(); }};
            bc.appendChild(b);
        }};
        mkBtn('◀', page - 1, page === 1, false);
        let sp = Math.max(1, page - 2), ep = Math.min(pages, sp + 4);
        if (ep - sp < 4) sp = Math.max(1, ep - 4);
        for (let i = sp; i <= ep; i++) mkBtn(i, i, false, i === page);
        mkBtn('▶', page + 1, page === pages, false);
    }}

    init();
}})();
</script>
"""

display(HTML(tabla_html))

## 5. Vista rápida con Pandas (alternativa estática)

Si preferís una vista más simple, acá tenés las primeras 20 filas del dataset con los nombres descriptivos:

In [ ]:
df_renombrado.head(20)

## 6. Estadísticas descriptivas

In [ ]:
df_renombrado.describe().round(2)